In [1]:
!pip install transformers datasets evaluate accelerate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
import json
from datasets import Dataset, DatasetDict

def load_squad_v2(path):
    with open(path) as f:
        data = json.load(f)
    examples = []
    for article in data['data']:
        for para in article['paragraphs']:
            context = para['context']
            for qa in para['qas']:
                question = qa['question']
                answer = "unanswerable" if qa['is_impossible'] else qa['answers'][0]['text']
                examples.append({"input": f"question: {question} context: {context}", "target": answer})
    return Dataset.from_list(examples)

train_path = "/kaggle/input/datasets/buildformacarov/squad-20/train-v2.0.json"
dev_path   = "/kaggle/input/datasets/buildformacarov/squad-20/dev-v2.0.json"
dataset = DatasetDict({"train": load_squad_v2(train_path), "validation": load_squad_v2(dev_path)})

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def preprocess(batch):
    model_inputs = tokenizer(batch["input"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(batch["target"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto", torch_dtype="auto")
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [6]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-base-lora-squadv2",
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, num_train_epochs=3,
    learning_rate=3e-4, fp16=True,
    eval_strategy="epoch", save_strategy="epoch",
    predict_with_generate=False, logging_steps=100,
    report_to="none", push_to_hub=False, dataloader_num_workers=0
)

trainer = Seq2SeqTrainer(model=model, args=training_args,
                         train_dataset=tokenized_dataset["train"],
                         eval_dataset=tokenized_dataset["validation"])
trainer.tokenizer = tokenizer

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

Epoch,Training Loss,Validation Loss


In [ ]:
def answer(question, context):
    text = f"question: {question} context: {context}"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    outputs = model.generate(**inputs, max_length=64)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer("What's the capital of France?", "France is a country in Europe. Its capital is Paris."))